CELDA 1 — Instalación de librerías

In [ ]:
!pip install tensorflow scikit-learn pymongo dnspython --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 2.8 MB/s eta 0:00:00


CELDA 2 — Librerías

In [ ]:
import numpy as np
import pandas as pd
from pymongo import MongoClient
from datetime import datetime, timezone
from google.colab import userdata

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, SimpleRNN, Bidirectional, Dense, Dropout

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CELDA 3 — Conexión a MongoDB

In [ ]:
TICKERS = ["FSM", "VOLCABC1.LM", "ABX.TO", "BVN", "BHP"]
VENTANA = 20        # días de contexto por secuencia
EPOCAS = 80
BATCH_SIZE = 64

MONGO_URI = userdata.get("MONGO_URI")
client = MongoClient(MONGO_URI)
db = client["ernesto_investing_ai"]

coleccion_precios = db["precios_ohlcv"]
coleccion_predicciones = db["predicciones"]
coleccion_metricas = db["metricas_modelos"]

try:
    client.admin.command("ping")
    print("✅ Conexión a MongoDB Atlas exitosa.")
except Exception as e:
    print("❌ Error de conexión:", e)

✅ Conexión a MongoDB Atlas exitosa.


CELDA 4 — Función: leer de Mongo y preparar secuencias

In [ ]:
def leer_precios_de_mongo(ticker):
    doc = coleccion_precios.find_one({"ticker": ticker})
    if doc is None:
        raise ValueError(f"No hay datos en Mongo para {ticker}. Corre primero el Notebook 1.")

    df = pd.DataFrame({
        "Close": doc["close"],
        "SMA20": doc["sma20"],
        "SMA50": doc["sma50"],
        "EMA12": doc["ema12"],
        "EMA26": doc["ema26"],
        "RSI": doc["rsi"],
        "MACD": doc["macd"],
    }, index=pd.to_datetime(doc["fechas"]))

    df.dropna(inplace=True)
    return df


def preparar_secuencias(df, ventana=VENTANA):
    """
    Crea secuencias deslizantes de 'ventana' días para clasificación binaria:
    1=BUY si el retorno del día siguiente a la ventana es positivo.
    """
    df = df.copy()
    df["Retorno"] = df["Close"].pct_change()
    df["Trend"] = np.where(df["Retorno"].shift(-1) > 0, 1, 0)
    df.dropna(inplace=True)

    features = ["Close", "SMA20", "SMA50", "EMA12", "EMA26", "RSI", "MACD"]
    scaler = MinMaxScaler()
    datos_norm = scaler.fit_transform(df[features])

    X, y = [], []
    for i in range(ventana, len(datos_norm)):
        X.append(datos_norm[i - ventana:i])
        y.append(df["Trend"].iloc[i])

    X = np.array(X)
    y = np.array(y)

    # Partición temporal 80/20 (sin mezclar orden)
    corte = int(len(X) * 0.8)
    X_train, X_test = X[:corte], X[corte:]
    y_train, y_test = y[:corte], y[corte:]

    return X_train, X_test, y_train, y_test, X, scaler, df

CELDA 5 — Arquitecturas de los 4 modelos

In [ ]:
def construir_lstm(input_shape):
    modelo = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(32),
        Dense(1, activation="sigmoid")
    ])
    modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return modelo


def construir_bilstm(input_shape):
    modelo = Sequential([
        Bidirectional(LSTM(50, return_sequences=True), input_shape=input_shape),
        Dropout(0.3),
        Bidirectional(LSTM(25)),
        Dense(1, activation="sigmoid")
    ])
    modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return modelo


def construir_gru(input_shape):
    modelo = Sequential([
        GRU(70, return_sequences=True, input_shape=input_shape),
        GRU(35),
        Dense(1, activation="sigmoid")
    ])
    modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return modelo


def construir_simplernn(input_shape):
    modelo = Sequential([
        SimpleRNN(45, return_sequences=True, input_shape=input_shape),
        SimpleRNN(22),
        Dense(1, activation="sigmoid")
    ])
    modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return modelo


ARQUITECTURAS = {
    "LSTM": construir_lstm,
    "BiLSTM": construir_bilstm,
    "GRU": construir_gru,
    "SimpleRNN": construir_simplernn
}

CELDA 6 — Función: entrenar un modelo y calcular métricas

In [ ]:
def entrenar_modelo_rnn(nombre_arquitectura, X_train, X_test, y_train, y_test):
    input_shape = (X_train.shape[1], X_train.shape[2])
    modelo = ARQUITECTURAS[nombre_arquitectura](input_shape)

    modelo.fit(
        X_train, y_train,
        epochs=EPOCAS,
        batch_size=BATCH_SIZE,
        verbose=0,
        validation_split=0.1
    )

    prob_pred = modelo.predict(X_test, verbose=0).flatten()
    pred = (prob_pred > 0.5).astype(int)

    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred, zero_division=0)
    recall = recall_score(y_test, pred, zero_division=0)
    f1 = f1_score(y_test, pred, zero_division=0)
    matriz = confusion_matrix(y_test, pred)

    return {
        "modelo_entrenado": modelo,
        "accuracy": round(float(accuracy), 4),
        "precision": round(float(precision), 4),
        "recall": round(float(recall), 4),
        "f1": round(float(f1), 4),
        "matriz_confusion": matriz.tolist()
    }

CELDA 7 — Función: guardar resultados en Mongo

In [ ]:
def guardar_resultados_rnn(ticker, nombre_arquitectura, resultado, senal_actual, confianza):
    ahora = datetime.now(timezone.utc)

    documento_prediccion = {
        "ticker": ticker,
        "modelo": nombre_arquitectura,
        "senal_actual": senal_actual,
        "confianza": confianza,
        "fecha_actualizacion": ahora
    }

    documento_metricas = {
        "ticker": ticker,
        "modelo": nombre_arquitectura,
        "accuracy": resultado["accuracy"],
        "precision": resultado["precision"],
        "recall": resultado["recall"],
        "f1": resultado["f1"],
        "matriz_confusion": resultado["matriz_confusion"],
        "hiperparametros": {
            "ventana": VENTANA,
            "epocas": EPOCAS,
            "batch_size": BATCH_SIZE
        },
        "fecha_actualizacion": ahora
    }

    coleccion_predicciones.update_one(
        {"ticker": ticker, "modelo": nombre_arquitectura},
        {"$set": documento_prediccion},
        upsert=True
    )

    coleccion_metricas.update_one(
        {"ticker": ticker, "modelo": nombre_arquitectura},
        {"$set": documento_metricas},
        upsert=True
    )

CELDA 8 — Ejecución: loop sobre 5 tickers × 4 arquitecturas

In [ ]:
resultados_completos = {}

for ticker in TICKERS:
    print(f"\n{'='*50}")
    print(f"Ticker: {ticker}")
    print(f"{'='*50}")

    try:
        df = leer_precios_de_mongo(ticker)
        X_train, X_test, y_train, y_test, X_completo, scaler, df_proc = preparar_secuencias(df)

        resultados_completos[ticker] = {}

        for nombre_arq in ARQUITECTURAS.keys():
            print(f"  Entrenando {nombre_arq}...")
            resultado = entrenar_modelo_rnn(nombre_arq, X_train, X_test, y_train, y_test)

            # Señal actual: predicción sobre la última secuencia disponible
            ultima_secuencia = X_completo[-1:].reshape(1, VENTANA, X_completo.shape[2])
            prob_actual = float(resultado["modelo_entrenado"].predict(ultima_secuencia, verbose=0)[0][0])
            senal_actual = "BUY" if prob_actual > 0.5 else "SELL"
            confianza = round(prob_actual if senal_actual == "BUY" else 1 - prob_actual, 4)

            guardar_resultados_rnn(ticker, nombre_arq, resultado, senal_actual, confianza)
            resultados_completos[ticker][nombre_arq] = resultado

            print(f"    {nombre_arq}: acc={resultado['accuracy']} f1={resultado['f1']} "
                  f"señal={senal_actual} ({confianza})")

        print(f"  ✅ {ticker} completado.")

    except Exception as e:
        print(f"  ❌ Error con {ticker}: {e}")


Ticker: FSM
  Entrenando LSTM...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


    LSTM: acc=0.4368 f1=0.6016 señal=BUY (0.5803)
  Entrenando BiLSTM...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


    BiLSTM: acc=0.5287 f1=0.6095 señal=BUY (0.6579)
  Entrenando GRU...


...

    BiLSTM: acc=0.4713 f1=0.5965 señal=SELL (0.576)
  Entrenando GRU...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


    GRU: acc=0.5517 f1=0.5618 señal=SELL (0.5122)
  Entrenando SimpleRNN...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


    SimpleRNN: acc=0.5057 f1=0.6718 señal=BUY (0.8785)
  ✅ BHP completado.


CELDA 9 — Verificación final

In [ ]:
print("Verificación — señales y métricas por ticker/modelo:\n")

for ticker in TICKERS:
    print(f"--- {ticker} ---")
    for doc in coleccion_predicciones.find(
        {"ticker": ticker, "modelo": {"$in": list(ARQUITECTURAS.keys())}},
        {"_id": 0}
    ):
        print(f"  {doc['modelo']}: {doc['senal_actual']} (confianza {doc['confianza']})")
    print()

Verificación — señales y métricas por ticker/modelo:

--- FSM ---
  LSTM: BUY (confianza 0.5803)
  BiLSTM: BUY (confianza 0.6579)
  GRU: BUY (confianza 0.6644)
  SimpleRNN: BUY (confianza 0.8586)

--- VOLCABC1.LM ---
  LSTM: BUY (confianza 0.5435)
  BiLSTM: BUY (confianza 0.5377)
  GRU: BUY (confianza 0.6343)
  SimpleRNN: BUY (confianza 0.6032)

--- ABX.TO ---
  LSTM: BUY (confianza 0.7369)
  BiLSTM: BUY (confianza 0.7576)
  GRU: BUY (confianza 0.769)
  SimpleRNN: BUY (confianza 0.5835)

--- BVN ---
  LSTM: BUY (confianza 0.7969)
  BiLSTM: BUY (confianza 0.8144)
  GRU: BUY (confianza 0.8616)
  SimpleRNN: BUY (confianza 0.5529)

--- BHP ---
  LSTM: SELL (confianza 0.6143)
  BiLSTM: SELL (confianza 0.576)
  GRU: SELL (confianza 0.5122)
  SimpleRNN: BUY (confianza 0.8785)

